In [4]:
# ==============================================================================
# SCRIPT DE CLASSIFICAÇÃO DE ARQUIVOS IFC
# ==============================================================================

# --- 1. Importações Necessárias ---
import os
import pandas as pd
import ifcopenshell
import ifcopenshell.util.element
import joblib
import json
import numpy as np # Adicionado para manipulação de dados

# --- 2. Configuração: Caminhos dos Arquivos ---
# Altere este caminho para o novo arquivo IFC que você quer classificar
IFC_FILE_PATH = r"C:\Users\lucas.galicioli\Downloads\RÔGGA EMPREENDIMENTOS-BRUSQUE HOME CLUB-2025-10-16-13-35-31-469\PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TORRE-R00.ifc"

# Caminhos para os artefatos do modelo (devem estar na mesma pasta ou com o caminho completo)
MODEL_PATH = 'modelo_disciplinas_v1.pkl'
ENCODER_PATH = 'label_encoder_disciplinas_v1.pkl'
COLUMNS_PATH = 'colunas_modelo_disciplinas.json'
MEDIANS_PATH = 'medianas_treinamento.json'

# --- 3. Carregamento dos Artefatos do Modelo ---
print("Carregando artefatos do modelo treinado...")
try:
    modelo = joblib.load(MODEL_PATH)
    label_encoder = joblib.load(ENCODER_PATH)
    with open(COLUMNS_PATH, 'r') as f:
        colunas_do_modelo = json.load(f)
    with open(MEDIANS_PATH, 'r') as f:
        medianas_treinamento = json.load(f)
    print("-> Artefatos carregados com sucesso!")
except FileNotFoundError as e:
    print(f"ERRO CRÍTICO: Não foi possível encontrar um dos arquivos do modelo: {e}")
    exit() # Encerra o script se os arquivos do modelo não forem encontrados

# --- 4. Funções de Extração de Dados do IFC (Seu código original) ---

def get_building_storey(element):
    try:
        spatial_container = ifcopenshell.util.element.get_container(element)
        if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
            return spatial_container.Name
    except Exception:
        pass
    return None

def get_material_name(element):
    material = ifcopenshell.util.element.get_material(element)
    if not material: return None
    if hasattr(material, 'Name'): return material.Name
    elif hasattr(material, 'MaterialLayers'):
        layer_names = [
            layer.Material.Name for layer in material.MaterialLayers 
            if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
        ]
        return ', '.join(layer_names) if layer_names else None
    return None
    
def get_quantity_value_legacy(element, quantity_name):
    for definition in getattr(element, 'IsDefinedBy', []):
        if definition.is_a('IfcRelDefinesByProperties'):
            prop_set = definition.RelatingPropertyDefinition
            if prop_set.is_a('IfcElementQuantity'):
                for quantity in prop_set.Quantities:
                    if quantity.Name == quantity_name:
                        value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
                        if value_attribute:
                            return getattr(quantity, value_attribute)
    return None

# --- 5. Processamento Principal: Extração e Classificação ---

try:
    # ETAPA A: Extrair dados do IFC para um DataFrame
    print(f"\nIniciando processamento do arquivo IFC: {os.path.basename(IFC_FILE_PATH)}...")
    ifc_file = ifcopenshell.open(IFC_FILE_PATH)
    products = ifc_file.by_type('IfcProduct')
    element_data = []

    for product in products:
        if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'):
            continue

        psets = ifcopenshell.util.element.get_psets(product)
        rogga_pset = psets.get('PSET_RÔGGA', {})

        element_info = {
            'Class': product.is_a(),
            'PredefinedType': getattr(product, 'PredefinedType', None),
            'Name': getattr(product, 'Name', None),
            'BuildingStorey': get_building_storey(product),
            'Material': get_material_name(product),
            'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
            'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
            'Width': get_quantity_value_legacy(product, 'Width'),
            'Thickness': get_quantity_value_legacy(product, 'Thickness'),
            'Length': get_quantity_value_legacy(product, 'Length'),
            'Height': get_quantity_value_legacy(product, 'Height'),
            'FileName': os.path.basename(IFC_FILE_PATH),
            'GlobalId': product.GlobalId
        }
        element_data.append(element_info)
    
    df_ifc_data = pd.DataFrame(element_data)
    print(f"-> {len(df_ifc_data)} elementos extraídos do IFC.")

    # ETAPA B: Pré-processar o DataFrame para o modelo (One-Hot Encoding e mais)
    print("\nIniciando pré-processamento dos dados...")
    
    #   1. Seleciona apenas as colunas que o modelo espera como entrada inicial
    features_selecionadas = [
        'Class', 'PredefinedType', 'BuildingStorey', 'Material',
        'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
        'Width', 'Thickness', 'Length', 'Height'
    ]
    df_para_prever = df_ifc_data[features_selecionadas].copy()

    #   2. Trata valores nulos (categóricos e numéricos) usando as mesmas regras do treino
    for col in df_para_prever.select_dtypes(include=['object']).columns:
        df_para_prever[col].fillna('Desconhecido', inplace=True)
    
    for col, mediana in medianas_treinamento.items():
        df_para_prever[col].fillna(mediana, inplace=True)

    #   3. Aplica One-Hot Encoding
    df_encodado = pd.get_dummies(df_para_prever)

    #   4. Limpa nomes das colunas
    df_encodado.columns = df_encodado.columns.str.replace(r'\[|\]|<', '_', regex=True)

    #   5. Alinha as colunas com o modelo (CRUCIAL!)
    df_final = df_encodado.reindex(columns=colunas_do_modelo, fill_value=0)
    print("-> Pré-processamento concluído.")

    # ETAPA C: Fazer as previsões
    print("\nRealizando previsões com o modelo...")
    previsoes_numericas = modelo.predict(df_final)
    previsoes_texto = label_encoder.inverse_transform(previsoes_numericas)
    print("-> Previsões realizadas com sucesso!")

    # ETAPA D: Apresentar o resultado
    df_ifc_data['Disciplina_Prevista'] = previsoes_texto

    print("\n--- AMOSTRA DO RESULTADO DA CLASSIFICAÇÃO ---")
    print(df_ifc_data[['Class', 'Name', 'Disciplina_Prevista']].head(20).fillna(''))

    # ETAPA E: Salvar o resultado completo em um novo arquivo CSV
    output_filename = f"classificado_{os.path.basename(IFC_FILE_PATH)}.csv"
    df_ifc_data.to_csv(output_filename, index=False, sep=';', decimal=',')
    print(f"\nResultado completo salvo em: {output_filename}")


except FileNotFoundError:
    print(f"ERRO: O arquivo IFC não foi encontrado em: {IFC_FILE_PATH}")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Carregando artefatos do modelo treinado...
-> Artefatos carregados com sucesso!

Iniciando processamento do arquivo IFC: PHN21043-CLI-EX-0002-BIM-TOR-GER-MODELO_DA_TORRE-R00.ifc...
-> 22109 elementos extraídos do IFC.

Iniciando pré-processamento dos dados...
-> Pré-processamento concluído.

Realizando previsões com o modelo...


C:\Users\lucas.galicioli\AppData\Local\Temp\ipykernel_21160\2514406971.py:122: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_para_prever[col].fillna('Desconhecido', inplace=True)
c:\Users\lucas.galicioli\ifc-classifier\.venv\Lib\site-packages\xgboost\core.py:729: UserWarning: [16:48:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, wh

-> Previsões realizadas com sucesso!

--- AMOSTRA DO RESULTADO DA CLASSIFICAÇÃO ---
                      Class  \
0   IfcBuildingElementProxy   
1   IfcBuildingElementProxy   
2   IfcBuildingElementProxy   
3   IfcBuildingElementProxy   
4   IfcBuildingElementProxy   
5   IfcBuildingElementProxy   
6   IfcBuildingElementProxy   
7   IfcBuildingElementProxy   
8   IfcBuildingElementProxy   
9   IfcBuildingElementProxy   
10  IfcBuildingElementProxy   
11  IfcBuildingElementProxy   
12  IfcBuildingElementProxy   
13  IfcBuildingElementProxy   
14  IfcBuildingElementProxy   
15  IfcBuildingElementProxy   
16  IfcBuildingElementProxy   
17  IfcBuildingElementProxy   
18  IfcBuildingElementProxy   
19  IfcBuildingElementProxy   

                                                 Name Disciplina_Prevista  
0   Identificador de Origem1:Identificador de Orig...   Impermeabilização  
1   ARC - SUPORTE PARA CONDENSADORAS:SUPORTE PARA ...   Impermeabilização  
2   ARC - SUPORTE PARA CONDENSADORAS